In [1]:
"""
SCALING-LAW COMPARISON
======================
Compares architectures by their loss-vs-parameters curve, not by single-point
performance. Whoever has the lowest, steepest curve scales best.

Architectures (chronological):
  Gen0:  Vanilla RNN              -- no substrate
  Gen1:  Single bounded substrate -- v2 single-sphere
  Gen3:  Composed s + 3p          -- central anchor + three orthogonal extensions

Sizes targeted (recurrent-param budget, NOT total params):
  small  ~  500
  medium ~ 2000
  large  ~ 8000
Hidden widths are picked so each generation lands near each target.

Same task (noisy sine), same optimizer (Adam lr=3e-3), same batch (16),
same seq_len (48), same train steps (300), same 3 seeds.

Run with:
   python3 scaling_compare.py

It prints a table and writes scaling_results.json. To make a plot
afterward you can run plot_scaling.py (separate, not required).

Wall-clock estimate: ~6-9 minutes on a typical laptop CPU. If the large
size is too slow you can comment it out at the bottom of this file.
"""
import math, json, time, numpy as np, torch
import torch.nn as nn

torch.set_num_threads(2)


# ---------- task ----------
def make_batch(B, T, device='cpu', noise=0.3):
    t = torch.linspace(0, 6*math.pi, T+1, device=device).unsqueeze(0).expand(B, -1)
    ph = torch.rand(B, 1, device=device) * 2*math.pi
    clean = torch.sin(t + ph)
    n = torch.randn_like(clean) * noise
    return (clean + n)[:, :-1].unsqueeze(-1), clean[:, 1:].unsqueeze(-1)


# ---------- architectures ----------

class Gen0_Vanilla(nn.Module):
    def __init__(self, H):
        super().__init__()
        self.rnn = nn.RNN(1, H, 1, nonlinearity='tanh', batch_first=True)
        self.head = nn.Linear(H, 1)
    def forward(self, x):
        return self.head(self.rnn(x)[0])
    @staticmethod
    def recurrent_params(H):
        return H * H            # weight_hh_l0 only

class Gen1_Single(nn.Module):
    def __init__(self, H, r=1.0):
        super().__init__()
        self.H, self.r = H, r
        self.inp = nn.Linear(1, H)
        self.W = nn.Linear(H, H, bias=False)
        self.head = nn.Linear(H, 1)
    def _bound(self, h):
        nrm = h.norm(dim=-1, keepdim=True).clamp_min(1e-6)
        return torch.tanh(nrm / self.r) * self.r * (h / nrm)
    def forward(self, x):
        B, T, _ = x.shape
        h = torch.zeros(B, self.H, device=x.device)
        outs = []
        for ti in range(T):
            h = self._bound(self.W(h) + self.inp(x[:, ti]))
            outs.append(self.head(h))
        return torch.stack(outs, 1)
    @staticmethod
    def recurrent_params(H):
        return H * H

class Gen3_Composed(nn.Module):
    """Central anchor s + three orthogonal p-extensions. Each carries an H-dim state.
    Recurrent params = 4 * H^2 (W_s + W_px + W_py + W_pz)."""
    def __init__(self, H, r=1.0, couple=0.3):
        super().__init__()
        self.H, self.r, self.couple = H, r, couple
        self.inp = nn.Linear(1, H)
        self.W_s = nn.Linear(H, H, bias=False)
        self.W_px = nn.Linear(H, H, bias=False)
        self.W_py = nn.Linear(H, H, bias=False)
        self.W_pz = nn.Linear(H, H, bias=False)
        self.head = nn.Linear(H, 1)
    def _bound(self, h):
        nrm = h.norm(dim=-1, keepdim=True).clamp_min(1e-6)
        return torch.tanh(nrm / self.r) * self.r * (h / nrm)
    def forward(self, x):
        B, T, _ = x.shape
        s = torch.zeros(B, self.H, device=x.device)
        px = torch.zeros_like(s); py = torch.zeros_like(s); pz = torch.zeros_like(s)
        outs = []
        for ti in range(T):
            u = self.inp(x[:, ti])
            px = self._bound(self.W_px(px) + u - self.couple * (px - s))
            py = self._bound(self.W_py(py) + u - self.couple * (py - s))
            pz = self._bound(self.W_pz(pz) + u - self.couple * (pz - s))
            s = self._bound(self.W_s(s) + (px + py + pz) / 3)
            outs.append(self.head(s))
        return torch.stack(outs, 1)
    @staticmethod
    def recurrent_params(H):
        return 4 * H * H


# ---------- pick hidden widths so each architecture lands near each target ----------
# Targets are *recurrent* param counts (the only fair thing to match).
TARGETS = [500, 2000, 8000]

def pick_H(arch_cls, target):
    # invert recurrent_params(H) ~= target  ->  pick smallest H whose count >= target
    H = 1
    while arch_cls.recurrent_params(H) < target:
        H += 1
    return H

SIZE_TABLE = {}
for arch in [Gen0_Vanilla, Gen1_Single, Gen3_Composed]:
    SIZE_TABLE[arch.__name__] = [(pick_H(arch, t), arch.recurrent_params(pick_H(arch, t)), t) for t in TARGETS]

print("=" * 70)
print("SIZE TABLE — H chosen so recurrent_params is closest to each target")
print("=" * 70)
print(f"{'arch':<22} {'target':>8} {'H':>5} {'actual_params':>15}")
for name, rows in SIZE_TABLE.items():
    for H, params, target in rows:
        print(f"{name:<22} {target:>8} {H:>5} {params:>15}")
print()


# ---------- training harness ----------
def train_once(model, steps=300, B=16, T=48, lr=3e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.MSELoss()
    losses = []
    t0 = time.time()
    for step in range(steps):
        x, y = make_batch(B, T)
        loss = lossf(model(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(float(loss.item()))
    wall = time.time() - t0
    final = float(np.mean(losses[-20:]))
    # also report best-loss-during-training so we don't penalize noisy late steps
    best = float(min(losses[20:]) if len(losses) > 20 else min(losses))
    return final, best, wall


# ---------- run the sweep ----------
SEEDS = [0, 1, 2]
ARCHS = [Gen0_Vanilla, Gen1_Single, Gen3_Composed]

results = {}
overall_t0 = time.time()

print("=" * 70)
print(f"SCALING SWEEP — 3 archs x 3 sizes x {len(SEEDS)} seeds = {3*3*len(SEEDS)} runs")
print("=" * 70)
for arch in ARCHS:
    name = arch.__name__
    results[name] = []
    for (H, params, target) in SIZE_TABLE[name]:
        finals, bests, walls = [], [], []
        for seed in SEEDS:
            torch.manual_seed(seed); np.random.seed(seed)
            model = arch(H)
            f, b, w = train_once(model)
            finals.append(f); bests.append(b); walls.append(w)
        row = {
            'H': H, 'recurrent_params': params, 'target': target,
            'final_loss_mean': float(np.mean(finals)),
            'final_loss_std':  float(np.std(finals)),
            'best_loss_mean':  float(np.mean(bests)),
            'best_loss_std':   float(np.std(bests)),
            'wall_mean':       float(np.mean(walls)),
            'seeds': SEEDS,
            'finals': finals, 'bests': bests, 'walls': walls,
        }
        results[name].append(row)
        print(f"  {name:<22} H={H:<4} params={params:<6} "
              f"final={row['final_loss_mean']:.4f}±{row['final_loss_std']:.4f}  "
              f"best={row['best_loss_mean']:.4f}  "
              f"wall={row['wall_mean']:.1f}s")
print()
print(f"total wall time: {time.time()-overall_t0:.1f}s")


# ---------- print scaling table ----------
print()
print("=" * 70)
print("SCALING TABLE  (best-loss vs recurrent-params, mean across seeds)")
print("=" * 70)
print(f"{'arch':<22} | " + " | ".join([f"~{t}p".rjust(14) for t in TARGETS]))
print("-" * 70)
for name, rows in results.items():
    cells = []
    for r in rows:
        cells.append(f"{r['best_loss_mean']:.4f}±{r['best_loss_std']:.4f}".rjust(14))
    print(f"{name:<22} | " + " | ".join(cells))

# slope per architecture: rough scaling exponent
# best_loss ~ A * params^(-alpha)  ->  log(loss) = log A - alpha * log(params)
print()
print("SCALING EXPONENT alpha  (loss ~ params^-alpha; bigger alpha = scales better)")
print("-" * 70)
for name, rows in results.items():
    xs = np.log([r['recurrent_params'] for r in rows])
    ys = np.log([r['best_loss_mean'] for r in rows])
    # linear fit
    slope = float(np.polyfit(xs, ys, 1)[0])
    alpha = -slope
    print(f"  {name:<22}  alpha = {alpha:+.3f}   "
          f"(loss at smallest size: {rows[0]['best_loss_mean']:.4f}, "
          f"at largest: {rows[-1]['best_loss_mean']:.4f})")

with open('scaling_results.json', 'w') as f:
    json.dump({'targets': TARGETS, 'size_table': {k: [list(t) for t in v]
              for k, v in SIZE_TABLE.items()}, 'results': results}, f, indent=2)
print("\nSaved scaling_results.json")
print("\nReading the result:")
print("  - 'alpha' is the scaling exponent. Bigger alpha = loss drops faster with size.")
print("  - The architecture with the highest alpha is the one that *scales* best.")
print("  - The architecture with the lowest absolute loss at the largest size is the one that *wins* at scale.")
print("  - If alphas are similar, the architectures scale equivalently and only the offsets differ.")
print("  - If one architecture's alpha is much larger, that's a real evolutionary advantage.")

SIZE TABLE — H chosen so recurrent_params is closest to each target
arch                     target     H   actual_params
Gen0_Vanilla                500    23             529
Gen0_Vanilla               2000    45            2025
Gen0_Vanilla               8000    90            8100
Gen1_Single                 500    23             529
Gen1_Single                2000    45            2025
Gen1_Single                8000    90            8100
Gen3_Composed               500    12             576
Gen3_Composed              2000    23            2116
Gen3_Composed              8000    45            8100

SCALING SWEEP — 3 archs x 3 sizes x 3 seeds = 27 runs
  Gen0_Vanilla           H=23   params=529    final=0.0293±0.0006  best=0.0237  wall=3.7s
  Gen0_Vanilla           H=45   params=2025   final=0.0284±0.0013  best=0.0225  wall=3.3s
  Gen0_Vanilla           H=90   params=8100   final=0.0287±0.0030  best=0.0211  wall=2.9s
  Gen1_Single            H=23   params=529    final=0.0272±0.0006  